In [1]:
backend = "gpu"

In [ ]:
import qutip
from qutip import *
import matplotlib.pyplot as plt
from typing import Union
from qutip import basis
from scipy import integrate

# to update parameters
import optax
if backend == "gpu":
	import jax
	import jax.numpy as np
	import qutip_jax
	qutip_jax.set_as_default()
	default_state_type = "jax"
	default_op_type = "jaxdia"
else:
	import numpy as np
	default_state_type = "CSR"
	default_op_type = "CSR"




In [3]:
#parameters
omega_r = 2 * np.pi * 50 #MHz
omega_max = omega_r
omega_R = 2.5 * omega_r #MHz #Applying on the target qubit
omega_c = 2 * np.pi * 50 #MHz #Applying on the control qubit
Dta = 2 * np.pi * 1200 #MHz #dutuning of the intermediate state
T_p = 16 * np.pi * Dta/(3 * omega_max**2) #target pulse duration
T_c = np.pi/omega_r #control pulse duration
tmax = T_c * 2 + T_p #total time of the simulation
V_dd = 2 * np.pi * 500 #MHz #dipole-dipole interaction strength
V_vdws = 2 * np.pi * 1 #MHz #van der Waals interaction strength
tau_c = 548 #mu s  #control qubit relaxation time
tau_p = 0.131 #mu s #target qubit relaxation time(intermediate state)
tau_R = 505 #mu s #target qubit relaxation time (Rydberg state)
gamma_c = 1/tau_c # control qubit decay rate
gamma_p = 1/tau_p # target qubit decay rate (intermediate state)
gamma_R = 1/tau_R # target qubit decay rate (Rydberg state)

In [4]:
#time list
len_mult = 1
tlist = np.linspace(0, len_mult*tmax, len_mult*1000)

with qutip.CoreOptions(default_dtype=default_state_type):
    #control atom states
    g = basis(3,0) #|0>  
    e = basis(3,1) #|1>   
    r = basis(3,2) #|r>

    #dagger
    g_dag = g.dag()  # <0|
    e_dag = e.dag()  # <1|
    r_dag = r.dag()  # <r|

    #target atom states
    A = basis(3,0) 
    B = basis(3,1)
    R = basis(3,2)

    #dagger
    A_dag = A.dag()
    B_dag = B.dag()
    R_dag = R.dag()

In [5]:
from functools import partial

In [6]:
#control field
# https://github.com/qutip/qutip-jax/issues/94, default document has bug
@jax.jit
def Omega_c_gr(t, omega_c, T_c, tmax, T_p):
    return np.heaviside(T_c-t, 1) * omega_c + np.heaviside(t-T_c-T_p, 1) * omega_c * np.heaviside(tmax-t, 1)
    #unknown function

@jax.jit
def Omega_c_er(t, omega_c, T_c, tmax, T_p):
    return np.heaviside(T_c-t, 1) * omega_c + np.heaviside(t-T_c-T_p, 1) * omega_c * np.heaviside(tmax-t, 1)
    #unknown function

In [7]:
#probe field(on target qubit)
@jax.jit
def Omega_p_AR(t, omega_c, T_c, tmax, T_p):
    #unknown function
    return np.heaviside(T_c-t, 1) * omega_c + np.heaviside(t-T_c-T_p, 1) * omega_c * np.heaviside(tmax-t, 1)

@jax.jit
def Omega_p_BR(t, omega_c, T_c, tmax, T_p):
    #unknown function
    return np.heaviside(T_c-t, 1) * omega_c + np.heaviside(t-T_c-T_p, 1) * omega_c * np.heaviside(tmax-t, 1)

In [8]:
with qutip.CoreOptions(default_dtype=default_op_type):
	# Hamiltonian operators
	id3 = Qobj(np.eye(3))
	id4 = Qobj(np.eye(4))
	#control Hamiltonian
	hc_op1 = (e * r_dag + r * e_dag)
	hc_op2 = (g * r_dag + r * g_dag)

	Hc_op1 = tensor(hc_op1, id3, id3) + tensor(id3, hc_op1, id3)
	Hc_op2 = tensor(hc_op2, id3, id3) + tensor(id3, hc_op2, id3) 
	# target Hamiltonian
	ht_op1 = (A * R_dag + R * A_dag)
	ht_op2 = (B * R_dag + R * B_dag)

	Ht_op1 = tensor(id3, id3, ht_op1) 
	Ht_op2 = tensor(id3, id3, ht_op2) 

	ht_dta = - Dta * R * R_dag
	Ht_dta = tensor(id3, id3, ht_dta)
	# interaction Hamiltonian
	hrr_op = r * r_dag
	hRR_op = R * R_dag
	HRr1 =  V_dd * tensor(hrr_op, id3, hRR_op)
	HRr2 =  V_dd * tensor(id3, hrr_op, hRR_op)
	HRr = HRr1 + HRr2


In [9]:
# decay operators
L_ce = np.sqrt(gamma_c) * (e * r_dag)
L_cg = np.sqrt(gamma_c) * (g * r_dag)
L_P = np.sqrt(gamma_R) * (A * R_dag)
L_R = np.sqrt(gamma_R) * (B * R_dag)

In [10]:
#total hamiltonian
Hnhermitian = [HRr, Ht_dta, [Hc_op1, qutip.coefficient(Omega_c_er,args={"omega_c":omega_c,"T_c":T_c,"tmax":tmax,"T_p":T_p})],[Hc_op2, qutip.coefficient(Omega_c_gr,args={"omega_c":omega_c,"T_c":T_c,"tmax":tmax,"T_p":T_p})], [Ht_op1, qutip.coefficient(Omega_p_AR,args={"omega_c":omega_c,"T_c":T_c,"tmax":tmax,"T_p":T_p})], [Ht_op2, qutip.coefficient(Omega_p_BR,args={"omega_c":omega_c,"T_c":T_c,"tmax":tmax,"T_p":T_p})]]

In [ ]:
# collapse operators
with qutip.CoreOptions(default_dtype=default_op_type):
      c_ops = tensor((L_cg+L_ce), id3, id3)\
            + tensor(id3, (L_cg+L_ce), id3) \
            + tensor(id3, id3, (L_R+L_P)) 

In [12]:
#expected states

with qutip.CoreOptions(default_dtype=default_state_type):	
	init_11 = tensor(g, g, A) #|00>_c |0>_t  #initial input state
	out_11 = tensor(g, g, A) #|00>_c |1>_t  #expected output state

	init_12 = tensor(g, g, B) #|00>_c |0>_t  #initial input state
	out_12 = tensor(g, g, B) #|00>_c |1>_t  #expected output state

	init_21 = tensor(g, e, A) #|01>_c |0>_t  #initial input state
	out_21 = tensor(g, e, B) #|01>_c |1>_t  #expected output state

	init_22 = tensor(g, e, B) #|01>_c |0>_t  #initial input state
	out_22 = tensor(g, e, A) #|01>_c |1>_t  #expected output state

	init_31 = tensor(e, g, A) #|01>_c |0>_t  #initial input state
	out_31 = tensor(e, g, B) #|01>_c |1>_t  #expected output state

	init_32 = tensor(e, g, B) #|01>_c |0>_t  #initial input state
	out_32 = tensor(e, g, A) #|01>_c |1>_t  #expected output state

	init_41 = tensor(e, e, A) #|11>_c |0>_t  #initial input state
	out_41 = tensor(e, e, A) #|11>_c |1>_t  #expected output state

	init_42 = tensor(e, e, B) #|11>_c |0>_t  #initial input state
	out_42 = tensor(e, e, B) #|11>_c |1>_t  #expected output state


In [13]:
from diffrax import PIDController, Tsit5
options = dict(
	method = "diffrax",
	solver = Tsit5(),
	# https://docs.kidger.site/diffrax/api/stepsize_controller/ parameters of PIDController for controlling the step size
	stepsize_controller = PIDController(rtol=1e-6, atol=1e-6,dtmax=T_c/2),
)

In [ ]:

# loss function, params: parameters that you want to optimize. 
def f(params):
	omega_c = params["omega_c"]
	T_c = params["T_c"]
	T_p = params["T_p"]
	tmax = params["tmax"]
	mc = mcsolve(Hnhermitian, init_11, tlist, c_ops, [init_11*init_11.dag(), out_11*out_11.dag()],args={"omega_c":omega_c,"T_c":T_c, "T_p":T_p, "tmax":tmax}, options=options,ntraj=500)
    # Return the expectation value of the number operator at the final time
	return mc.expect[0][-1].real


In [ ]:
import optax
# specify optimizer to do gradient descent
optimizer = optax.adam(learning_rate=0.1)
# specify initial parameters
params = {
	"omega_c": 2 * np.pi * 50,  # MHz
	"T_c": T_c,  # control pulse duration
	"T_p": T_p,  # target pulse duration
	"tmax": tmax,  # total time of the simulation
} 
# initialize optimizer state
opt_state = optimizer.init(params)

In [ ]:
for _ in range(1000):
  grads = jax.grad(f)(params)
  updates, opt_state = optimizer.update(grads, opt_state)
  params = optax.apply_updates(params, updates)

/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/qutip/solver/solver_base.py:576: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(


20.0%. Run time:   0.00s. Est. time left: 00:00:00:00


/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/equinox/_jit.py:55: UserWarning: Complex dtype support in Diffrax is a work in progress and may not yet produce correct results. Consider splitting your computation into real and imaginary parts instead.
  out = fun(*args, **kwargs)


40.0%. Run time:  27.97s. Est. time left: 00:00:00:41
60.0%. Run time:  47.29s. Est. time left: 00:00:00:31
80.0%. Run time:  66.35s. Est. time left: 00:00:00:16
100.0%. Run time:  86.66s. Est. time left: 00:00:00:00
Total run time: 105.36s
20.0%. Run time:   0.00s. Est. time left: 00:00:00:00
40.0%. Run time:  24.03s. Est. time left: 00:00:00:36
60.0%. Run time:  42.71s. Est. time left: 00:00:00:28
80.0%. Run time:  62.62s. Est. time left: 00:00:00:15
100.0%. Run time:  80.47s. Est. time left: 00:00:00:00
Total run time: 100.13s
20.0%. Run time:   0.00s. Est. time left: 00:00:00:00
40.0%. Run time:  24.27s. Est. time left: 00:00:00:36
60.0%. Run time:  44.21s. Est. time left: 00:00:00:29
80.0%. Run time:  63.40s. Est. time left: 00:00:00:15
100.0%. Run time:  81.32s. Est. time left: 00:00:00:00
Total run time: 101.05s
20.0%. Run time:   0.00s. Est. time left: 00:00:00:00
40.0%. Run time:  25.61s. Est. time left: 00:00:00:38
60.0%. Run time:  45.67s. Est. time left: 00:00:00:30
80.0%. 

In [1]:
import qutip_jax
import qutip
import jax
import jax.numpy as jnp
from functools import partial
from qutip import mcsolve

# Use JAX backend for QuTiP
qutip_jax.set_as_default()

# Define time-dependent functions
@jax.jit
def H_1_coeff(t, omega):
    return 2.0 * jnp.pi * 0.25 * jnp.cos(2.0 * omega * t)

# Define operators and states
size = 10
a = qutip.tensor(qutip.destroy(size), qutip.qeye(2)).to('jaxdia')    # Annihilation operator
sm = qutip.qeye(size).to('jaxdia') & qutip.sigmax().to('jaxdia')  # Example spin operator

# Define the Hamiltonian
H_0 = 2.0 * jnp.pi * a.dag() * a + 2.0 * jnp.pi * sm.dag() * sm
H_1_op = sm * a.dag() + sm.dag() * a

# Initialize the Hamiltonian with time-dependent coefficients
H = [H_0, [H_1_op, qutip.coefficient(H_1_coeff, args={"omega": 1.0})]]

# Define initial states, mixed states are not supported
state = qutip.basis(size, size - 1).to('jax') & qutip.basis(2, 1).to('jax')

# Define collapse operators and observables
c_ops = [jnp.sqrt(0.1) * a]
e_ops = [a.dag() * a, sm.dag() * sm]

# Time list
tlist = jnp.linspace(0.0, 10.0, 101)

# Define the function for which we want to compute the gradient
def f(omega):
    result = mcsolve(
        H, state, tlist, c_ops, e_ops, ntraj=10,
        args={"omega": omega},
        options={"method": "diffrax"}
    )
    # Return the expectation value of the number operator at the final time
    return result.expect[0][-1].real

# Compute the gradient
# gradient = jax.grad(f)(1.0)

In [ ]:
import optax

In [ ]:
optimizer = optax.adam(learning_rate=0.1)
param = 0.1
opt_state = optimizer.init(param)

In [6]:
# https://github.com/google-deepmind/optax/blob/main/docs/getting_started.ipynb to use optax to compute optimal control fields
for _ in range(1000):
  grads = jax.grad(f)(param)
  updates, opt_state = optimizer.update(grads, opt_state)
  param = optax.apply_updates(param, updates)


10.0%. Run time:   0.00s. Est. time left: 00:00:00:00


/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/qutip/solver/solver_base.py:576: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(
/home/ubuntu/WandaPulse/.venv/lib/python3.10/site-packages/equinox/_jit.py:55: UserWarning: Complex dtype support in Diffrax is a work in progress and may not yet produce correct results. Consider splitting your computation into real and imaginary parts instead.
  out = fun(*args, **kwargs)


20.0%. Run time:  14.67s. Est. time left: 00:00:00:58
30.0%. Run time:  19.87s. Est. time left: 00:00:00:46
40.0%. Run time:  22.41s. Est. time left: 00:00:00:33
50.0%. Run time:  24.49s. Est. time left: 00:00:00:24
60.0%. Run time:  27.10s. Est. time left: 00:00:00:18
70.0%. Run time:  29.36s. Est. time left: 00:00:00:12
80.0%. Run time:  31.66s. Est. time left: 00:00:00:07
90.0%. Run time:  33.88s. Est. time left: 00:00:00:03
100.0%. Run time:  36.13s. Est. time left: 00:00:00:00
Total run time:  38.86s
10.0%. Run time:   0.00s. Est. time left: 00:00:00:00
20.0%. Run time:  11.29s. Est. time left: 00:00:00:45
30.0%. Run time:  15.37s. Est. time left: 00:00:00:35
40.0%. Run time:  17.62s. Est. time left: 00:00:00:26
50.0%. Run time:  21.16s. Est. time left: 00:00:00:21
60.0%. Run time:  24.81s. Est. time left: 00:00:00:16
70.0%. Run time:  27.06s. Est. time left: 00:00:00:11
80.0%. Run time:  29.26s. Est. time left: 00:00:00:07
90.0%. Run time:  32.32s. Est. time left: 00:00:00:03
100

KeyboardInterrupt: 